**This notebook was used to test out the isolation of grandparent/parent nodes with the effective isolation of also their own children nodes as well as the update and saving process of each of these nodes. In here you can test out the call the elements of the subroutines/functions inside by defining inside them on 'subroutine_key'.**

In [1]:
%reload_ext autoreload
%autoreload 2
from fparser.two import Fortran2003 as F23
from fparser.two import Fortran2008 as F28
from fparser.two.utils import walk
import os
from typing import Dict, List,Tuple,Any
from collections import deque
import yaml
import ast
import re
import itertools

In [2]:
%cd ..

/home/ssivanes/Fgpt


/data/ssivanes/fparser-venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
# Processor class
from processor import Processor
from extractor import Extractor
from isolator import Isolator
processor = Processor()

INFO     Processor initialized.

In [4]:
rest_of_path = "/data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/"
target_module = "hydrol" # "hydrol" explicitsnow
work = os.getenv("work")

In [5]:
isolator = Isolator(rest_of_path, target_module, work,False)

cls = Extractor(isolator.module_dir_sp, isolator.module_tree_sp)
cls.find_subroutines()
cls.extract_loop_indices()

INFO     Processor initialized.

INFO     Successfully parsed file:                                                                                 
         /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/hydrol_org.f90

INFO     Successfully parsed string!

INFO     Processor initialized.

WARNING  Subroutine 'hydrol_main' calls 'explicitsnow_main' which is not defined in current module

...

INFO     Successfully parsed file:                                                                                 
         /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/albedo_surface.f90

INFO     Successfully parsed file:                                                                                 
         /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/explicitsnow_org.f90

INFO     Found subroutine 'explicitsnow_main' in file:                                                             
         /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/explicitsnow.f90

INFO     Backup file already exists:                                                                               
         /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/explicitsnow_org.f90

INFO     Found external subroutine 'explicitsnow_main' in file:                                                    
         /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/explicitsnow.f90, adding to processing queue

In [6]:
cls.subroutine_keys_all

{'explicitsnow_age',
 'explicitsnow_compactn',
 'explicitsnow_compactn_up',
 'explicitsnow_drift',
 'explicitsnow_fall',
 'explicitsnow_gone',
 'explicitsnow_grain',
 'explicitsnow_icelevels',
 'explicitsnow_icemelt',
 'explicitsnow_iceprofile',
 'explicitsnow_levels',
 'explicitsnow_main',
 'explicitsnow_maxmass',
 'explicitsnow_melt_refrz',
 'explicitsnow_profile',
 'explicitsnow_subli',
 'explicitsnow_transf',
 'hydrol_alma',
 'hydrol_canop',
 'hydrol_diag_soil',
 'hydrol_diag_soil_flux',
 'hydrol_flood',
 'hydrol_hydraulic_arch_tuzet_calc',
 'hydrol_hydraulic_arch_tuzet_muff',
 'hydrol_hydraulic_arch_tuzet_resist',
 'hydrol_main',
 'hydrol_muff_radial_coef_setup',
 'hydrol_muff_radial_resolution',
 'hydrol_nudge_mc',
 'hydrol_nudge_mc_diag',
 'hydrol_nudge_snow',
 'hydrol_root_profile',
 'hydrol_soil',
 'hydrol_soil_coef',
 'hydrol_soil_froz',
 'hydrol_soil_infilt',
 'hydrol_soil_setup',
 'hydrol_soil_smooth_over_mcs',
 'hydrol_soil_smooth_over_mcs2',
 'hydrol_soil_smooth_under_mcr

In [7]:
cls.subroutine_keys_ncl

{'explicitsnow_age',
 'explicitsnow_compactn',
 'explicitsnow_compactn_up',
 'explicitsnow_drift',
 'explicitsnow_fall',
 'explicitsnow_gone',
 'explicitsnow_grain',
 'explicitsnow_icelevels',
 'explicitsnow_icemelt',
 'explicitsnow_iceprofile',
 'explicitsnow_levels',
 'explicitsnow_profile',
 'explicitsnow_subli',
 'explicitsnow_transf',
 'hydrol_alma',
 'hydrol_canop',
 'hydrol_diag_soil',
 'hydrol_flood',
 'hydrol_muff_radial_resolution',
 'hydrol_nudge_mc',
 'hydrol_soil_coef',
 'hydrol_soil_froz',
 'hydrol_soil_setup',
 'hydrol_soil_smooth_over_mcs',
 'hydrol_soil_smooth_over_mcs2',
 'hydrol_soil_smooth_under_mcr',
 'hydrol_soil_tridiag'}

In [8]:
# FInding the parents but also using the 
print(cls.subroutine_keys_all - cls.subroutine_keys_ncl)

{'hydrol_soil', 'hydrol_nudge_mc_diag', 'hydrol_diag_soil_flux', 'hydrol_tmc_update', 'hydrol_hydraulic_arch_tuzet_calc', 'hydrol_hydraulic_arch_tuzet_muff', 'explicitsnow_main', 'hydrol_muff_radial_coef_setup', 'hydrol_soil_infilt', 'hydrol_root_profile', 'hydrol_main', 'hydrol_nudge_snow', 'hydrol_vegupd', 'explicitsnow_maxmass', 'hydrol_split_soil', 'hydrol_hydraulic_arch_tuzet_resist', 'explicitsnow_melt_refrz'}


In [9]:
cls.subroutines.keys()

dict_keys(['hydrol_main', 'hydrol_tmc_update', 'hydrol_canop', 'hydrol_vegupd', 'hydrol_flood', 'hydrol_soil', 'hydrol_soil_infilt', 'hydrol_soil_smooth_under_mcr', 'hydrol_soil_smooth_over_mcs', 'hydrol_soil_smooth_over_mcs2', 'hydrol_diag_soil_flux', 'hydrol_soil_tridiag', 'hydrol_soil_coef', 'hydrol_soil_froz', 'hydrol_soil_setup', 'hydrol_split_soil', 'hydrol_diag_soil', 'hydrol_alma', 'hydrol_nudge_mc', 'hydrol_nudge_mc_diag', 'hydrol_nudge_snow', 'hydrol_hydraulic_arch_tuzet_calc', 'hydrol_hydraulic_arch_tuzet_resist', 'hydrol_hydraulic_arch_tuzet_muff', 'hydrol_muff_radial_coef_setup', 'hydrol_muff_radial_resolution', 'hydrol_root_profile', 'explicitsnow_main', 'explicitsnow_grain', 'explicitsnow_compactn', 'explicitsnow_compactn_up', 'explicitsnow_drift', 'explicitsnow_transf', 'explicitsnow_fall', 'explicitsnow_gone', 'explicitsnow_melt_refrz', 'explicitsnow_icemelt', 'explicitsnow_icelevels', 'explicitsnow_levels', 'explicitsnow_profile', 'explicitsnow_iceprofile', 'explicits

In [10]:
%reload_ext autoreload
%autoreload 2
from transformer import Transformer
from utils import identify_replace_all
import time

In [11]:
transformer = Transformer("/home/ssivanes/Fgpt/benchmark",isolator,cls,None,config_path = "/home/ssivanes/Fgpt/template.yaml")

╭───────────────────────────────────────── Fortran → Python Transformer ──────────────────────────────────────────╮
│ 🚀 Starting Module: Transformer                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────── Fortran → Python Transformer ──────────────────────────────────────────╮
│ 🚀 Starting Module: F2NP                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [12]:
subroutine_key = 'hydrol_hydraulic_arch_tuzet_muff' # explicitsnow_main hydrol_vegupd hydrol_soil hydrol_flood hydrol_alma hydrol_canop hydrol_hydraulic_arch_tuzet_calc

In [13]:
isolator.parent_subroutine_call = set()
for child_procedure in ['hydrol_hydraulic_arch_tuzet_muff']:
    isolator.isolate_procedure(cls, transformer,'hydrol_hydraulic_arch_tuzet_calc',child_procedure)

INFO       Call site 1: CALL hydrol_hydraulic_arch_tuzet_muff(kjit, kjpindex, ipts, ivm, soiltile, veget_max, njsc,
         ks, nvan, avan, F_absorption_temp, circ_class_biomass, circ_class_n, Res_root_sup, Res_root_inf,          
         Fsup_temp, Finf_temp, psi_root_sup_temp, psi_root_inf_temp, mc_sup_temp, mc_inf_temp, mc_i_sup_temp,      
         mc_i_inf_temp)

WARNING  Warning: Implicit shape detected in the declaration!

WARNING  Node: REAL(KIND = r_std), DIMENSION(:, :), INTENT(IN) :: soiltile

INFO     Processor initialized.

INFO     Corresponding element of "soiltile" is "soiltile" in call statement in subroutine                         
         "hydrol_hydraulic_arch_tuzet_calc"!

...

WARNING  Missing file: /home/ssivanes/Fgpt/benchmark/hydrol_hydraulic_arch_tuzet_muff/dummy.bin

WARNING  Missing file: /home/ssivanes/Fgpt/benchmark/hydrol_hydraulic_arch_tuzet_muff/global.bin

WARNING  Benchmark files do not exist yet. Run the modified main code and then python executive.py

INFO     Successfully wrote code to file: /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/hydrol.f90

INFO     Successfully wrote code to file: /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/hydrol.f90

In [14]:
isolator.working_subroutines.keys() # This has all the currently working subroutines upon the currecly working subroutines, which means 
# if we have call statement or function call inside the current working subroutines, they are added onto the working_subroutines dict. 

dict_keys(['hydrol_muff_radial_resolution', 'hydrol_muff_radial_coef_setup', 'hydrol_hydraulic_arch_tuzet_muff'])

In [15]:
print(list(cls.dec_global[subroutine_key])) # THis will now contain all the global values that will be used inside the 
# Parent subroutine and global values of the child subroutines. that we retrieve using this isolator.collect_global_vars_decl(cls.dec_global[child_subroutine_key], cls.dec_global[subroutine_key])
# which modifies direclty the cls.dec_global[subroutine_key] of the parent declarations 

['pref_soil_veg', 'r_froot', 'zlt', 'un', 'humcste', 'iroot', 'icarbon', 'min_sechiba', 'lim_layer', 'srl', 'dh', 'kilo_to_unit', 'vegtot', 'test_pft', 'pi', 'mcr_sup', 'mcs_sup', 'mcr_inf', 'mcs_inf', 'dt_sechiba', 'printlev', 'numout', 'test_grid', 'min_stomate', 'is_vg', 'cte_grav', 'rho_h2o', 'mega_to_unit', 'psi_air_entry', 'b_muff', 'ncirc', 'nparts', 'nelements', 'nrp', 'ngrnd', 'nscm', 'nscm_usda', 'w_time', 'one_day', 'deux', 'zero', 'trois', 'huit']


In [16]:
# Most of the elements are the retrieved variables are the same for the children as for the parent. With the exception being is that all
# subroutines and functions are placed inside the global class of each node(children or parent) as well as the declaration_initalization of 
# their attributes and the main file which is a simple python script will contain all the class and method calling. This is helpful for the 
# the rest of the process on the auto differenciation process. 

In [18]:
#declaration_stmts = list(cls.dec_global[subroutine_key].values())
#ast_nodes = transformer.convert_SPECIFICATION_PART(declaration_stmts,cls_mode=True)

In [19]:
tree = transformer.update_global_python(subroutine_key,cls_mode = True,for_loop=True)
# THis the global module for the parent subroutine itself

10:41:17 [UPDATE] — Updating Global Python

INFO     Combined statement: INTEGER(KIND = i_std), DIMENSION(nvm) :: pref_soil_veg

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(nvm) :: r_froot

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(ngrnd) :: zlt

INFO     Successfully removed INTENT and SAVE attributes from statements

...

[INFO] Since no index is given, WILL BE USING previous known ast Assign position with this method: __init__

[INFO] Since argument method_name is: __init__, placing the assign statement inside of this method

[INFO] Since argument method_name is: __init__, placing the assign statement inside of this method

[INFO] Since no index is given, WILL BE USING previous known ast Assign position with this method: __init__

[INFO] Since argument method_name is: __init__, placing the assign statement inside of this method

In [20]:
print(ast.unparse(tree)) # ['is_tuzet_hydrol_arch']

import numpy as np
import logging
from scipy.io import FortranFile
import os

class Global_module_hydrol_hydraulic_arch_tuzet_muff:

    def __init__(self):
        self.nice = np.int32(8)
        self.nsnow = np.int32(3)
        self.nslm = np.int32(11)
        self.nvm = np.int32(15)
        self.nstm = np.int32(3)
        self.kjpindex = np.int32(4717)
        self.ier = np.int32(0)
        self.ic0 = np.int32(0)
        self.ic = np.int32(0)
        self.icr = np.float64(0.0)
        self.start_time = np.float64(0.0)
        self.stop_time = np.float64(0.0)
        self.un = np.float64(1.0)
        self.iroot = np.int32(6)
        self.icarbon = np.int32(1)
        self.min_sechiba = np.float64(1e-08)
        self.lim_layer = np.int32(9)
        self.kilo_to_unit = np.float64(1000.0)
        self.test_pft = np.int32(2)
        self.pi = np.float64(3.141592653589793)
        self.printlev = np.int32(2)
        self.numout = np.int32(6)
        self.test_grid = np.int32(1)
        se

In [21]:
print(transformer.dependant_variables)

{'zlt': ['ngrnd']}


In [22]:
isolator.processor.reads_in_decleration_routine

[Execution_Part(Read_Stmt(Io_Control_Spec_List(',', (Io_Control_Spec(None, Int_Literal_Constant('1363', None)), Io_Control_Spec('IOSTAT', Name('ier')))), None, Input_Item_List(',', (Name('dt_sechiba'),))), If_Construct(If_Then_Stmt(Level_4_Expr(Name('ier'), '/=', Int_Literal_Constant('0', None))), Write_Stmt(Io_Control_Spec_List(',', (Io_Control_Spec(None, Io_Unit('*')), Io_Control_Spec(None, Format('*')))), Output_Item_List(',', (Char_Literal_Constant("'Error reading from file for dt_sechiba. '", None), Char_Literal_Constant("' IOSTAT : '", None), Name('ier')))), End_If_Stmt('IF', None))),
 Execution_Part(Read_Stmt(Io_Control_Spec_List(',', (Io_Control_Spec(None, Int_Literal_Constant('1363', None)), Io_Control_Spec('IOSTAT', Name('ier')))), None, Input_Item_List(',', (Name('is_vg'),))), If_Construct(If_Then_Stmt(Level_4_Expr(Name('ier'), '/=', Int_Literal_Constant('0', None))), Write_Stmt(Io_Control_Spec_List(',', (Io_Control_Spec(None, Io_Unit('*')), Io_Control_Spec(None, Format('*')

In [23]:
print(walk(walk(isolator.processor.reads_in_read_routine,F23.Input_Item_List),F23.Name))

[Name('b_muff'), Name('dh'), Name('humcste'), Name('mcr_inf'), Name('mcr_sup'), Name('mcs_inf'), Name('mcs_sup'), Name('pref_soil_veg'), Name('psi_air_entry'), Name('r_froot'), Name('srl'), Name('vegtot'), Name('zlt')]


In [24]:
# Where statements 
subroutine_code = """
subroutine compute_c(a, b, c)
  ! Main WHERE block
  WHERE (a > 0)               
    c = b * 2.0   

    WHERE (b < 3.0)      
      c = b + 10.0           
    ELSEWHERE (b >= 3.0)      
      c = b - 1.0            
    END WHERE                 

  ELSEWHERE (a < 0)         
    c = -b                

  ELSEWHERE              
    c = 0.0           

  END WHERE        

end subroutine compute_c
"""

subroutine_code = """
subroutine compute_c(a, b, c, n)
  implicit none
  integer, intent(in) :: n
  real, intent(in) :: a(n), b(n)
  real, intent(out) :: c(n)
  integer :: i

  ! Example loop to process in chunks or apply conditionally
  do i = 1, n
    if (mod(i, 2) == 0) then
      ! Main WHERE block for even indices
      WHERE (a > 0.0)
        c = b * 2.0

        WHERE (b < 3.0)
          c = b + 10.0
        ELSEWHERE (b >= 3.0)
          c = b - 1.0
        END WHERE

      ELSEWHERE (a < 0.0)
        c = -b

      ELSEWHERE
        c = 0.0
      END WHERE

    else
      ! For odd indices, maybe apply a different logic
      WHERE (a < 0.0)
        c = -b * 2.0
      ELSEWHERE
        c = b
      END WHERE
    end if
  end do

end subroutine compute_c
"""
sub_parser = processor.parse_fortran_string(subroutine_code)


INFO     Successfully parsed string!

In [25]:
from f2np import F2NP
f2np = F2NP(cls)

╭───────────────────────────────────────── Fortran → Python Transformer ──────────────────────────────────────────╮
│ 🚀 Starting Module: F2NP                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [26]:
# _,_,subroutine_code_ast = f2np.recursive_ast(sub_parser)

In [27]:
# print(ast.unparse(ast.fix_missing_locations(subroutine_code_ast[0])))

In [28]:
cls.scalar_variables[subroutine_key]

[Name('igrid'),
 Name('ipft'),
 Name('kjit'),
 Name('un'),
 Name('iroot'),
 Name('icarbon'),
 Name('min_sechiba'),
 Name('lim_layer'),
 Name('kilo_to_unit'),
 Name('test_pft'),
 Name('pi'),
 Name('dt_sechiba'),
 Name('printlev'),
 Name('numout'),
 Name('test_grid'),
 Name('min_stomate'),
 Name('is_vg'),
 Name('cte_grav'),
 Name('rho_h2o'),
 Name('mega_to_unit')]

In [29]:
main_tree = transformer.update_main_python(out_module=tree,subroutine_key=subroutine_key)

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Successfully removed INTENT and SAVE attributes from statements

...

[INFO] Inserting after last assign at position 21 inside the function : main

[INFO] Inserting after last assign at position 22 inside the function : main

[INFO] Inserting after last assign at position 23 inside the function : main

[INFO] Inserting after last assign at position 24 inside the function : main

[INFO] Inserting after last assign at position 25 inside the function : main

In [30]:
print(ast.unparse(main_tree))

import os
import time
import functools
import numpy as np
import logging
from scipy.io import FortranFile
from module_global import Global_module_hydrol_hydraulic_arch_tuzet_muff

def read_dummy(gmhhatm, avan, circ_class_biomass, circ_class_n, F_abs, igrid, ipft, kjit, ks, mc_i_inf_temp, mc_i_sup_temp, mc_inf_temp, mc_sup_temp, njsc, nvan, psi_root_inf_temp, psi_root_sup_temp, Res_root_inf, Res_root_sup, soiltile, veget_max):
    print(f'--- inside the read dummy routine for hydrol_hydraulic_arch_tuzet_muff ---')
    path = f'/home/ssivanes/Fgpt/benchmark/hydrol_hydraulic_arch_tuzet_muff/dummy.bin'
    ffile = FortranFile(path, 'r')
    avan[:] = ffile.read_reals(np.float64).reshape((gmhhatm.kjpindex,), order='F')
    circ_class_biomass[:] = ffile.read_reals(np.float64).reshape((gmhhatm.kjpindex, gmhhatm.nvm, gmhhatm.ncirc, gmhhatm.nparts, gmhhatm.nelements), order='F')
    circ_class_n[:] = ffile.read_reals(np.float64).reshape((gmhhatm.kjpindex, gmhhatm.nvm, gmhhatm.ncirc), order='F')

In [31]:
#transformer.transfer_to_pyfile(main_tree,subroutine_key,python_file_type="main")

In [32]:
#transformer.transfer_to_pyfile(tree,subroutine_key)